<h1 style="font-size: 1.6rem; font-weight: bold">ITO5163: Introduction to Cybersecurity</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 2: Symmetric Crytosystems Part 2</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **Feistel Cipher**

The **Feistel Cipher** is not one specific cipher, but a **Structure** for building block ciphers. Many real ciphers use it, including **DES**, Blowfish and Twofish. It splits a block of $2w$ bits into two halves and processes them over **Multiple Rounds**. Each round uses a different **Subkey** $K_i$ generated from the main key $K$. **16 Rounds** is typical (DES uses 16).

Each round **Substitutes** one half (using a round function $F$ and XOR) and then **Swaps** the halves. Repeating substitution and permutation many times like this is called a **Product Cipher**.

#### **One Round of Encryption**

```mermaid
flowchart TB
    L0["LEᵢ₋₁"] ~~~ R0["REᵢ₋₁"]
    R0 --> F["F"]
    K(["Kᵢ"]) --> F
    F --> X(("⊕"))
    L0 --> X
    R0 --> L1["LEᵢ = REᵢ₋₁"]
    X --> R1["REᵢ = LEᵢ₋₁ ⊕ F(REᵢ₋₁, Kᵢ)"]
```

$$LE_i = RE_{i-1}$$

$$RE_i = LE_{i-1} \oplus F(RE_{i-1}, K_i)$$

- **Each Round**:
  - The **round function** $F$ takes the right half and the subkey $K_i$
  - Its output is **XORed** with the left half to make the new right half
  - The old right half becomes the new left half, **unchanged**
  - So each round only changes one half, and the next round changes the other

- **After the Last Round**: the two halves are **swapped** once more to give the ciphertext
  - $LE_{17} = RE_{16}$ and $RE_{17} = LE_{16}$

#### **Encryption and Decryption**

**Decryption uses the exact same process**, just with the **Subkeys in Reverse Order**. This works because XOR undoes itself: if $RE_i = LE_{i-1} \oplus F(RE_{i-1}, K_i)$, then XORing with the same $F$ output again gives back $LE_{i-1}$.

```mermaid
flowchart LR
    P["Plaintext"] --> R1["Round 1<br>K₁"] --> R2["Round 2<br>K₂"] --> D1["..."] --> R16["Round 16<br>K₁₆"] --> W1["Swap<br>halves"] --> C["Ciphertext"]
    C2["Ciphertext"] --> Q1["Round 1<br>K₁₆"] --> Q2["Round 2<br>K₁₅"] --> D2["..."] --> Q16["Round 16<br>K₁"] --> W2["Swap<br>halves"] --> P2["Plaintext"]
```

- **Why Feistel is Useful**:
  - The same hardware or code does both encryption and decryption
  - **$F$ does not need to be reversible**. Decryption never runs $F$ backwards, it just runs $F$ forwards again and XORs. This means $F$ can be as complex as needed
  - At each step of decryption, the values match the encryption values with the halves swapped: $LD_i = RE_{16-i}$ and $RD_i = LE_{16-i}$

#### **Feistel Execution for "Hello!" with 16 Rounds**

The 6-byte block is split into two 3-byte halves (shown in hex). Values are from the Python code below.

| Round | Encryption $L$ | Encryption $R$ | Decryption $L$ | Decryption $R$ |
|-------|----------------|----------------|----------------|----------------|
| Start | 48656c         | 6c6f21         | 206d88         | 82d4be         |
| 1     | 6c6f21         | 883673         | 82d4be         | 643382         |
| 2     | 883673         | efbeb7         | 643382         | 95ae24         |
| ...   | ...            | ...            | ...            | ...            |
| 14    | 95ae24         | 643382         | efbeb7         | 883673         |
| 15    | 643382         | 82d4be         | 883673         | 6c6f21         |
| 16    | 82d4be         | 206d88         | 6c6f21         | 48656c         |
| Swap  | **206d88**     | **82d4be**     | **48656c**     | **6c6f21**     |

Final result:
- Ciphertext: `206d8882d4be`
- Decryption gives back `48656c 6c6f21` = "Hello!"
- Decryption round 1 matches encryption round 15 with the halves swapped, round 2 matches round 14, and so on

#### **Design Features**

| Feature                     | Larger or More Complex Means...   | Trade-off                    |
|-----------------------------|-----------------------------------|------------------------------|
| Block size                  | More secure                       | Slower                       |
| Key size                    | More secure                       | Slower                       |
| Number of rounds            | More secure                       | Slower                       |
| Subkey generation algorithm | Harder to cryptanalyse            | More complex to design       |
| Round function $F$          | Harder to cryptanalyse            | More complex to design       |

- **Other Design Goals**:
  - **Fast software encryption**: the cipher should run efficiently in normal code
  - **Ease of analysis**: a simple, clear design is easier to check for weaknesses


In [ ]:
# Feistel cipher in Python, rewritten from the GeeksforGeeks version:
# https://www.geeksforgeeks.org/feistel-cipher/
# For learning only. Do not use this to protect real data.
import hashlib

ROUNDS = 16


def generate_subkeys(key, rounds=ROUNDS):
    """Derive a different subkey K_i for every round from the main key K."""
    return [hashlib.sha256(key + bytes([i])).digest() for i in range(1, rounds + 1)]


def F(right, subkey):
    """Round function: mixes the right half with the subkey.
    It does NOT need to be reversible, since Feistel decryption never undoes F."""
    return hashlib.sha256(right + subkey).digest()[:len(right)]


def xor(a, b):
    return bytes(x ^ y for x, y in zip(a, b))


def feistel(block, subkeys, trace=False):
    """Run the Feistel rounds. The same function encrypts and decrypts:
    only the order of the subkeys changes."""
    half = len(block) // 2
    L, R = block[:half], block[half:]
    if trace:
        print(f"  Start    L={L.hex()}  R={R.hex()}")
    for i, k in enumerate(subkeys, start=1):
        L, R = R, xor(L, F(R, k))       # LE_i = RE_(i-1),  RE_i = LE_(i-1) XOR F(RE_(i-1), K_i)
        if trace:
            print(f"  Round {i:2}  L={L.hex()}  R={R.hex()}")
    return R + L                         # final swap


def encrypt(plaintext, key, trace=False):
    if len(plaintext) % 2:               # the block must split into two equal halves
        plaintext += b"\x00"
    return feistel(plaintext, generate_subkeys(key), trace)


def decrypt(ciphertext, key, trace=False):
    plaintext = feistel(ciphertext, generate_subkeys(key)[::-1], trace)  # subkeys in reverse
    return plaintext.rstrip(b"\x00")


# Driver code
key = b"my secret key"
plaintext = b"Hello!"

print("Plain text:     ", plaintext)
print("Encryption:")
ciphertext = encrypt(plaintext, key, trace=True)
print("Cipher text:    ", ciphertext.hex())
print("Decryption:")
print("Retrieved text: ", decrypt(ciphertext, key, trace=True))

<br>

***

### **Data Encryption Standard (DES) and Triple DES (3DES)**

**DES** uses the **Data Encryption Algorithm (DEA)**. It became the US national standard in **1977** and was widely used, especially by government. It was replaced by **AES** in 2001. DES is a **Feistel Block Cipher** that encrypts **64-bit Blocks** using a **56-bit Key**.

- **DES at a Glance**:
  - Block size: 64 bits
  - Key: 64 bits supplied, but only **56 bits** are used. Every 8th bit is a parity bit and is dropped
  - Rounds: 16
  - Subkeys: 16 subkeys of 48 bits, one per round

#### **DES Overview**

```mermaid
flowchart TB
    PT["64-bit plaintext"] --> IP["Initial permutation (IP)"] --> R1["Round 1"] --> R2["Round 2"] --> RD["..."] --> R16["Round 16"] --> SW["32-bit swap"] --> IPI["Inverse initial permutation (IP⁻¹)"] --> CT["64-bit ciphertext"]
    KEY["64-bit key"] --> PC1["Permuted choice 1<br>(drops 8 parity bits)"]
    PC1 -- "56" --> S1["Left circular shift"] -- "56" --> S2["Left circular shift"] --> SD["..."] --> S16["Left circular shift"]
    S1 --> P1["Permuted choice 2"] -- "K₁ (48)" --> R1
    S2 --> P2["Permuted choice 2"] -- "K₂ (48)" --> R2
    S16 --> P16["Permuted choice 2"] -- "K₁₆ (48)" --> R16
```

DES is a **Feistel Cipher** with two extra steps: a permutation at the **start** and its inverse at the **end**.

- **Encryption (left side)**:
  - **Initial permutation (IP)**: rearranges the 64 bits of **each** plaintext block
  - **16 rounds**: each one is a Feistel round using subkey $K_i$
  - **32-bit swap**: swaps the left and right halves after round 16
  - **Inverse initial permutation (IP⁻¹)**: undoes the IP to give the 64-bit ciphertext

- **Key Schedule (right side)**:
  - **Permuted choice 1 (PC-1)**: drops the 8 parity bits and rearranges the rest into 56 bits
  - Splits them into two 28-bit halves, $C$ and $D$
  - Each round: **left circular shift** both halves by 1 or 2 bits
  - **Permuted choice 2 (PC-2)**: picks 48 of the 56 bits to make the subkey $K_i$

#### **A Single DES Round**

```mermaid
flowchart TB
    L0["Lᵢ₋₁ (32 bits)"] ~~~ R0["Rᵢ₋₁ (32 bits)"]
    subgraph F["Round function F"]
        direction TB
        EX["Expansion E<br>32 → 48 bits"] --> X1(("⊕")) --> SB["8 S-boxes<br>48 → 32 bits"] --> PP["Permutation P<br>32 → 32 bits"]
    end
    R0 --> EX
    K(["Kᵢ (48 bits)"]) --> X1
    PP --> X2(("⊕"))
    L0 --> X2
    X2 --> Ri["Rᵢ"]
    R0 --> Li["Lᵢ = Rᵢ₋₁"]
```

$$L_i = R_{i-1}$$

$$R_i = L_{i-1} \oplus F(R_{i-1}, K_i)$$

- **Round Function $F$**:
  - **Expansion E**: expands the 32-bit right half to 48 bits by repeating some bits
  - **XOR** with the 48-bit subkey $K_i$
  - **S-boxes**: 8 S-boxes, each turning **6 bits into 4 bits**, giving 32 bits
    - The outer 2 bits of each group pick the row, and the middle 4 bits pick the column
    - This is the only **non-linear** step in DES, and it's where most of its security comes from
    - S-boxes are **not reversible** (6 bits in, 4 bits out), which is fine because Feistel never runs $F$ backwards
  - **Permutation P**: rearranges the 32 bits

- **Decryption**: the same algorithm with the subkeys in **reverse order** ($K_{16}$ first, $K_1$ last)

#### **DES Execution for Plaintext `123456ABCD132536`, Key `AABB09182736CCDD`**

Values are from the Python code below.

| Step          | $L$        | $R$        | Subkey $K_i$   |
|---------------|------------|------------|----------------|
| After IP      | 14A7D678   | 18CA18AD   | -              |
| Round 1       | 18CA18AD   | 5A78E394   | 194CD072DE8C   |
| Round 2       | 5A78E394   | 4A1210F6   | 4568581ABCCE   |
| ...           | ...        | ...        | ...            |
| Round 15      | BD2DD2AB   | CF26B472   | 3330C5D9A36D   |
| Round 16      | CF26B472   | 19BA9212   | 181C5D75C66D   |

Final result:
- After the 32-bit swap and IP⁻¹: ciphertext `C0B7A8D05F3A829C`

#### **Avalanche Effect**

A good cipher shows the **Avalanche Effect**: changing **one bit** of the plaintext (or key) changes about **half the bits** of the ciphertext.

| Plaintext            | Ciphertext           |
|----------------------|----------------------|
| `123456ABCD132536`   | `C0B7A8D05F3A829C`   |
| `123456ABCD132537`   | `F334195281AF7BA9`   |

Final result:
- 1 plaintext bit changed → **33 of 64** ciphertext bits changed (ideal is about 32)

#### **Security of DES**

- **Strengths**:
  - Strong design, with no practical shortcut attack found in decades of analysis
  - Strong avalanche effect
  - The S-boxes turned out to resist **differential cryptanalysis**, an attack that wasn't publicly known until 1990

- **Weaknesses**:
  - **56-bit key is too short**: only $2^{56}$ possible keys
  - In **1998**, the EFF's purpose-built "DES Cracker" found a DES key in under 3 days by brute force
  - The S-box **design criteria were kept secret** at first, which raised concerns about hidden weaknesses. They were published in 1994

#### **Double DES and Triple DES (3DES)**

Since the DES algorithm itself was strong, the fix was to run it **several times with different keys**.

**Double DES** encrypts twice: $C = E(K_2, E(K_1, P))$. The key is 112 bits, but it only gives about **57 bits** of security because of the **meet-in-the-middle attack**. The attacker encrypts $P$ forwards under every $K_1$, decrypts $C$ backwards under every $K_2$, and looks for a match in the middle. This takes about $2 \times 2^{56}$ steps instead of $2^{112}$.

**Triple DES** with two keys uses **Encrypt-Decrypt-Encrypt (EDE)**:

```mermaid
flowchart LR
    P["P"] --> E1["E<br>K₁"] -- "A" --> D2["D<br>K₂"] -- "B" --> E3["E<br>K₁"] --> C["C"]
    C2["C"] --> D1["D<br>K₁"] -- "B" --> E2["E<br>K₂"] -- "A" --> D3["D<br>K₁"] --> P2["P"]
```

$$C = E(K_1, D(K_2, E(K_1, P))) \qquad P = D(K_1, E(K_2, D(K_1, C)))$$

- **Why Decrypt in the Middle?**:
  - If $K_1 = K_2$, the first two steps cancel out, leaving single DES
  - This lets 3DES systems still talk to older DES systems

| Variant              | Keys          | Key Length | Effective Security |
|----------------------|---------------|------------|--------------------|
| DES                  | $K$           | 56 bits    | 56 bits            |
| Double DES           | $K_1, K_2$    | 112 bits   | About 57 bits (meet-in-the-middle) |
| 3DES (two keys)      | $K_1, K_2, K_1$ | 112 bits | About 80 bits      |
| 3DES (three keys)    | $K_1, K_2, K_3$ | 168 bits | About 112 bits (meet-in-the-middle) |

- **3DES Today**:
  - Three times slower than DES
  - Still uses a small 64-bit block
  - **Deprecated**: NIST disallowed it for encryption after 2023. AES is used instead

In [1]:
# DES and 3DES in Python, rewritten from the Java version on GeeksforGeeks:
# https://www.geeksforgeeks.org/data-encryption-standard-des-set-1/
# For learning only. DES and 3DES are no longer secure.

# Initial permutation and its inverse
IP = [58, 50, 42, 34, 26, 18, 10, 2, 60, 52, 44, 36, 28, 20, 12, 4,
      62, 54, 46, 38, 30, 22, 14, 6, 64, 56, 48, 40, 32, 24, 16, 8,
      57, 49, 41, 33, 25, 17, 9, 1, 59, 51, 43, 35, 27, 19, 11, 3,
      61, 53, 45, 37, 29, 21, 13, 5, 63, 55, 47, 39, 31, 23, 15, 7]
IP_INV = [40, 8, 48, 16, 56, 24, 64, 32, 39, 7, 47, 15, 55, 23, 63, 31,
          38, 6, 46, 14, 54, 22, 62, 30, 37, 5, 45, 13, 53, 21, 61, 29,
          36, 4, 44, 12, 52, 20, 60, 28, 35, 3, 43, 11, 51, 19, 59, 27,
          34, 2, 42, 10, 50, 18, 58, 26, 33, 1, 41, 9, 49, 17, 57, 25]

# Key schedule: PC-1 (64 -> 56 bits, drops parity bits), PC-2 (56 -> 48 bits)
PC1 = [57, 49, 41, 33, 25, 17, 9, 1, 58, 50, 42, 34, 26, 18,
       10, 2, 59, 51, 43, 35, 27, 19, 11, 3, 60, 52, 44, 36,
       63, 55, 47, 39, 31, 23, 15, 7, 62, 54, 46, 38, 30, 22,
       14, 6, 61, 53, 45, 37, 29, 21, 13, 5, 28, 20, 12, 4]
PC2 = [14, 17, 11, 24, 1, 5, 3, 28, 15, 6, 21, 10,
       23, 19, 12, 4, 26, 8, 16, 7, 27, 20, 13, 2,
       41, 52, 31, 37, 47, 55, 30, 40, 51, 45, 33, 48,
       44, 49, 39, 56, 34, 53, 46, 42, 50, 36, 29, 32]
SHIFTS = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]

# Round function: expansion E (32 -> 48 bits) and permutation P (32 -> 32 bits)
E = [32, 1, 2, 3, 4, 5, 4, 5, 6, 7, 8, 9, 8, 9, 10, 11,
     12, 13, 12, 13, 14, 15, 16, 17, 16, 17, 18, 19, 20, 21, 20, 21,
     22, 23, 24, 25, 24, 25, 26, 27, 28, 29, 28, 29, 30, 31, 32, 1]
P = [16, 7, 20, 21, 29, 12, 28, 17, 1, 15, 23, 26, 5, 18, 31, 10,
     2, 8, 24, 14, 32, 27, 3, 9, 19, 13, 30, 6, 22, 11, 4, 25]

# 8 S-boxes: each maps 6 bits -> 4 bits
SBOX = [
    [[14, 4, 13, 1, 2, 15, 11, 8, 3, 10, 6, 12, 5, 9, 0, 7], [0, 15, 7, 4, 14, 2, 13, 1, 10, 6, 12, 11, 9, 5, 3, 8],
     [4, 1, 14, 8, 13, 6, 2, 11, 15, 12, 9, 7, 3, 10, 5, 0], [15, 12, 8, 2, 4, 9, 1, 7, 5, 11, 3, 14, 10, 0, 6, 13]],
    [[15, 1, 8, 14, 6, 11, 3, 4, 9, 7, 2, 13, 12, 0, 5, 10], [3, 13, 4, 7, 15, 2, 8, 14, 12, 0, 1, 10, 6, 9, 11, 5],
     [0, 14, 7, 11, 10, 4, 13, 1, 5, 8, 12, 6, 9, 3, 2, 15], [13, 8, 10, 1, 3, 15, 4, 2, 11, 6, 7, 12, 0, 5, 14, 9]],
    [[10, 0, 9, 14, 6, 3, 15, 5, 1, 13, 12, 7, 11, 4, 2, 8], [13, 7, 0, 9, 3, 4, 6, 10, 2, 8, 5, 14, 12, 11, 15, 1],
     [13, 6, 4, 9, 8, 15, 3, 0, 11, 1, 2, 12, 5, 10, 14, 7], [1, 10, 13, 0, 6, 9, 8, 7, 4, 15, 14, 3, 11, 5, 2, 12]],
    [[7, 13, 14, 3, 0, 6, 9, 10, 1, 2, 8, 5, 11, 12, 4, 15], [13, 8, 11, 5, 6, 15, 0, 3, 4, 7, 2, 12, 1, 10, 14, 9],
     [10, 6, 9, 0, 12, 11, 7, 13, 15, 1, 3, 14, 5, 2, 8, 4], [3, 15, 0, 6, 10, 1, 13, 8, 9, 4, 5, 11, 12, 7, 2, 14]],
    [[2, 12, 4, 1, 7, 10, 11, 6, 8, 5, 3, 15, 13, 0, 14, 9], [14, 11, 2, 12, 4, 7, 13, 1, 5, 0, 15, 10, 3, 9, 8, 6],
     [4, 2, 1, 11, 10, 13, 7, 8, 15, 9, 12, 5, 6, 3, 0, 14], [11, 8, 12, 7, 1, 14, 2, 13, 6, 15, 0, 9, 10, 4, 5, 3]],
    [[12, 1, 10, 15, 9, 2, 6, 8, 0, 13, 3, 4, 14, 7, 5, 11], [10, 15, 4, 2, 7, 12, 9, 5, 6, 1, 13, 14, 0, 11, 3, 8],
     [9, 14, 15, 5, 2, 8, 12, 3, 7, 0, 4, 10, 1, 13, 11, 6], [4, 3, 2, 12, 9, 5, 15, 10, 11, 14, 1, 7, 6, 0, 8, 13]],
    [[4, 11, 2, 14, 15, 0, 8, 13, 3, 12, 9, 7, 5, 10, 6, 1], [13, 0, 11, 7, 4, 9, 1, 10, 14, 3, 5, 12, 2, 15, 8, 6],
     [1, 4, 11, 13, 12, 3, 7, 14, 10, 15, 6, 8, 0, 5, 9, 2], [6, 11, 13, 8, 1, 4, 10, 7, 9, 5, 0, 15, 14, 2, 3, 12]],
    [[13, 2, 8, 4, 6, 15, 11, 1, 10, 9, 3, 14, 5, 0, 12, 7], [1, 15, 13, 8, 10, 3, 7, 4, 12, 5, 6, 11, 0, 14, 9, 2],
     [7, 11, 4, 1, 9, 12, 14, 2, 0, 6, 10, 13, 15, 3, 5, 8], [2, 1, 14, 7, 4, 10, 8, 13, 15, 12, 9, 0, 3, 5, 6, 11]],
]


# ---------- bit helpers: a block is a list of 0/1 bits ----------
def hex_to_bits(h):
    return [int(b) for b in bin(int(h, 16))[2:].zfill(len(h) * 4)]


def bits_to_hex(bits):
    return hex(int("".join(map(str, bits)), 2))[2:].upper().zfill(len(bits) // 4)


def permute(bits, table):
    """Output bit i is input bit table[i] (tables count from 1)."""
    return [bits[t - 1] for t in table]


def xor(a, b):
    return [x ^ y for x, y in zip(a, b)]


def rotate_left(bits, n):
    return bits[n:] + bits[:n]


# ---------- key schedule ----------
def generate_subkeys(key_hex):
    """64-bit key -> PC-1 -> split into C, D (28 bits each) -> 16 rounds of shift + PC-2."""
    key = permute(hex_to_bits(key_hex), PC1)
    C, D = key[:28], key[28:]
    subkeys = []
    for shift in SHIFTS:
        C, D = rotate_left(C, shift), rotate_left(D, shift)
        subkeys.append(permute(C + D, PC2))  # 48-bit subkey K_i
    return subkeys


# ---------- round function F ----------
def sbox_substitute(bits48):
    """Split into eight 6-bit groups. Outer 2 bits pick the row, middle 4 bits pick the column."""
    out = []
    for i in range(8):
        b = bits48[i * 6:(i + 1) * 6]
        row = b[0] * 2 + b[5]
        col = b[1] * 8 + b[2] * 4 + b[3] * 2 + b[4]
        out += [int(x) for x in format(SBOX[i][row][col], "04b")]
    return out


def F(R, K):
    """Expand 32 -> 48, XOR with subkey, S-boxes 48 -> 32, permute P."""
    return permute(sbox_substitute(xor(permute(R, E), K)), P)


# ---------- DES ----------
def des(block_hex, subkeys, trace=False):
    bits = permute(hex_to_bits(block_hex), IP)          # initial permutation
    L, R = bits[:32], bits[32:]
    if trace:
        print(f"  After IP:  L0={bits_to_hex(L)}  R0={bits_to_hex(R)}")
    for i, K in enumerate(subkeys, start=1):
        L, R = R, xor(L, F(R, K))                       # Feistel round
        if trace:
            print(f"  Round {i:2}:  L={bits_to_hex(L)}  R={bits_to_hex(R)}  K={bits_to_hex(K)}")
    return bits_to_hex(permute(R + L, IP_INV))          # 32-bit swap, then inverse IP


def encrypt(plaintext_hex, key_hex, trace=False):
    return des(plaintext_hex, generate_subkeys(key_hex), trace)


def decrypt(ciphertext_hex, key_hex, trace=False):
    return des(ciphertext_hex, generate_subkeys(key_hex)[::-1], trace)  # same steps, keys reversed


# ---------- 3DES (EDE with two keys: K1, K2, K1) ----------
def triple_encrypt(plaintext_hex, k1, k2):
    return encrypt(decrypt(encrypt(plaintext_hex, k1), k2), k1)


def triple_decrypt(ciphertext_hex, k1, k2):
    return decrypt(encrypt(decrypt(ciphertext_hex, k1), k2), k1)


# ---------- avalanche effect ----------
def bits_changed(a_hex, b_hex):
    return bin(int(a_hex, 16) ^ int(b_hex, 16)).count("1")


# Driver code
plaintext = "123456ABCD132536"
key = "AABB09182736CCDD"

print("Encryption:")
ciphertext = encrypt(plaintext, key, trace=True)
print("Cipher text:", ciphertext)

print("\nDecryption:")
print("Plain text: ", decrypt(ciphertext, key, trace=True))

print("\nAvalanche effect:")
flipped = "123456ABCD132537"  # last bit of the plaintext flipped
c2 = encrypt(flipped, key)
print(f"  {plaintext} -> {ciphertext}")
print(f"  {flipped} -> {c2}")
print(f"  1 plaintext bit changed -> {bits_changed(ciphertext, c2)} of 64 ciphertext bits changed")

print("\n3DES (K1, K2, K1):")
k2 = "0123456789ABCDEF"
c3 = triple_encrypt(plaintext, key, k2)
print("  Cipher text:", c3)
print("  Plain text: ", triple_decrypt(c3, key, k2))
print("  With K1 = K2 it is the same as single DES:", triple_encrypt(plaintext, key, key) == ciphertext)

Encryption:
  After IP:  L0=14A7D678  R0=18CA18AD
  Round  1:  L=18CA18AD  R=5A78E394  K=194CD072DE8C
  Round  2:  L=5A78E394  R=4A1210F6  K=4568581ABCCE
  Round  3:  L=4A1210F6  R=B8089591  K=06EDA4ACF5B5
  Round  4:  L=B8089591  R=236779C2  K=DA2D032B6EE3
  Round  5:  L=236779C2  R=A15A4B87  K=69A629FEC913
  Round  6:  L=A15A4B87  R=2E8F9C65  K=C1948E87475E
  Round  7:  L=2E8F9C65  R=A9FC20A3  K=708AD2DDB3C0
  Round  8:  L=A9FC20A3  R=308BEE97  K=34F822F0C66D
  Round  9:  L=308BEE97  R=10AF9D37  K=84BB4473DCCC
  Round 10:  L=10AF9D37  R=6CA6CB20  K=02765708B5BF
  Round 11:  L=6CA6CB20  R=FF3C485F  K=6D5560AF7CA5
  Round 12:  L=FF3C485F  R=22A5963B  K=C2C1E96A4BF3
  Round 13:  L=22A5963B  R=387CCDAA  K=99C31397C91F
  Round 14:  L=387CCDAA  R=BD2DD2AB  K=251B8BC717D0
  Round 15:  L=BD2DD2AB  R=CF26B472  K=3330C5D9A36D
  Round 16:  L=CF26B472  R=19BA9212  K=181C5D75C66D
Cipher text: C0B7A8D05F3A829C

Decryption:
  After IP:  L0=19BA9212  R0=CF26B472
  Round  1:  L=CF26B472  R=BD2DD2AB  

<br>

***

### **Advanced Encryption Standard (AES)**

**AES** is the block cipher used today to protect commercial and government data. It was chosen in **2001** to replace DES. It uses a **128-bit Block** and supports three key sizes: **128, 192 or 256 bits**.

Unlike DES, AES is **not a Feistel Cipher**. Every round changes the **whole block**, not just one half. Each round uses four kinds of operation:

| Operation                | AES Step      |
|--------------------------|---------------|
| Byte substitution        | SubBytes      |
| Permutation              | ShiftRows     |
| Finite field arithmetic  | MixColumns    |
| XOR with a key           | AddRoundKey   |

#### **Finite Field $GF(2^8)$**

Cryptographic algorithms work with integers. Addition, subtraction and multiplication are easy, but **division** needs every nonzero number to have a **multiplicative inverse**. That only works in a **Finite Field**. AES uses $GF(2^8)$, where every **byte** is an element of the field.

Each byte is treated as a **polynomial** with binary coefficients, of degree at most 7:

$$f(x) = a_7x^7 + a_6x^6 + \cdots + a_1x + a_0 = \sum_{i=0}^{7} a_i x^i, \qquad a_i \in \{0, 1\}$$

For example, the byte `0x57` = `01010111` = $x^6 + x^4 + x^2 + x + 1$. There are $2^8 = 256$ such polynomials, one for each byte.

- **Arithmetic in $GF(2^8)$**:
  - **Addition**: add coefficients mod 2, which is just **XOR**
    - `0x57` ⊕ `0x83` = `0xD4`
  - **Multiplication**: multiply the polynomials, then take the **remainder** after dividing by a fixed **irreducible polynomial**
  - **AES's irreducible polynomial**: $m(x) = x^8 + x^4 + x^3 + x + 1$
    - **Irreducible** means it can't be written as the product of two polynomials of lower degree, the polynomial version of a prime number
    - Reducing by $m(x)$ keeps every result within 8 bits, and guarantees every nonzero byte has an inverse

#### **$GF(2^8)$ Multiplication Execution for `0x57` × `0x83`**

| Step | Working | Result |
|------|---------|--------|
| 1. Write as polynomials | `0x57` = $x^6 + x^4 + x^2 + x + 1$, `0x83` = $x^7 + x + 1$ | |
| 2. Multiply (coefficients mod 2) | $(x^6 + x^4 + x^2 + x + 1)(x^7 + x + 1)$ | $x^{13} + x^{11} + x^9 + x^8 + x^6 + x^5 + x^4 + x^3 + 1$ |
| 3. Reduce mod $m(x)$ | Remainder after dividing by $x^8 + x^4 + x^3 + x + 1$ | $x^7 + x^6 + 1$ |
| 4. Back to a byte | `11000001` | `0xC1` |

#### **The State**

The 16-byte block is arranged as a **4×4 Matrix of Bytes** called the **State**, filled **column by column**. Every step changes the state, and after the last round the state is the ciphertext. The key is expanded into round keys, which are also arranged as 4×4 grids.

For plaintext `00 11 22 33 44 55 66 77 88 99 aa bb cc dd ee ff`:

|           | Col 0 | Col 1 | Col 2 | Col 3 |
|-----------|-------|-------|-------|-------|
| **Row 0** | 00    | 44    | 88    | cc    |
| **Row 1** | 11    | 55    | 99    | dd    |
| **Row 2** | 22    | 66    | aa    | ee    |
| **Row 3** | 33    | 77    | bb    | ff    |

#### **AES Overview**

The number of rounds depends on the key size:

| Key Size           | Rounds | Expanded Key (32-bit words) |
|--------------------|--------|-----------------------------|
| 128 bits (16 bytes) | 10    | 44                          |
| 192 bits (24 bytes) | 12    | 52                          |
| 256 bits (32 bytes) | 14    | 60                          |

```mermaid
flowchart TB
    PT["Plaintext (16 bytes)"] --> ST["4×4 state"] --> AR0["AddRoundKey"]
    AR0 --> R1["Round 1<br>SubBytes → ShiftRows → MixColumns → AddRoundKey"]
    R1 --> RD["Rounds 2 to 9<br>(same 4 steps)"]
    RD --> R10["Round 10<br>SubBytes → ShiftRows → AddRoundKey<br>(no MixColumns)"]
    R10 --> CT["Ciphertext (16 bytes)"]
    KEY["Key (16 bytes)"] --> KE["Key expansion<br>44 words w[0..43]"]
    KE -- "w[0..3]" --> AR0
    KE -- "w[4..7]" --> R1
    KE -- "w[8..39]" --> RD
    KE -- "w[40..43]" --> R10
```

- **AES-128 Structure**:
  - **Initial transformation**: AddRoundKey with round key 0, before round 1
  - **Rounds 1 to 9**: all four steps
  - **Round 10**: three steps, with no MixColumns
  - **Key expansion**: the 16-byte key becomes 44 words of 32 bits, so 4 words (one 4×4 round key) for each of the 11 AddRoundKey steps

#### **The Four Round Steps**

```mermaid
flowchart LR
    S0["State"] --> SB["SubBytes<br>S-box on each byte"] --> SR["ShiftRows<br>rotate row r left by r"] --> MC["MixColumns<br>each column × matrix in GF(2⁸)"] --> AK["AddRoundKey<br>XOR with round key"] --> S1["State"]
```

- **SubBytes**: each byte is replaced using a fixed 16×16 lookup table, the **S-box**
  - The S-box is built from the **inverse in $GF(2^8)$**, followed by a fixed bit transformation
  - This is the only **non-linear** step, and it's where most of AES's resistance to attacks comes from
  - e.g. S-box[`0x53`] = `0xED`

- **ShiftRows**: each row is rotated **left** by its row number
  - Spreads the bytes of each column across all four columns

| Row | Before          | Shift  | After           |
|-----|-----------------|--------|-----------------|
| 0   | $s_{00}\ s_{01}\ s_{02}\ s_{03}$ | 0      | $s_{00}\ s_{01}\ s_{02}\ s_{03}$ |
| 1   | $s_{10}\ s_{11}\ s_{12}\ s_{13}$ | 1 left | $s_{11}\ s_{12}\ s_{13}\ s_{10}$ |
| 2   | $s_{20}\ s_{21}\ s_{22}\ s_{23}$ | 2 left | $s_{22}\ s_{23}\ s_{20}\ s_{21}$ |
| 3   | $s_{30}\ s_{31}\ s_{32}\ s_{33}$ | 3 left | $s_{33}\ s_{30}\ s_{31}\ s_{32}$ |

- **MixColumns**: each column is multiplied by a fixed matrix, using $GF(2^8)$ arithmetic
  - Every output byte depends on all four bytes of the column
  - Combined with ShiftRows, a change to one byte spreads to the whole state within two rounds

$$\begin{pmatrix} s'_{0} \\ s'_{1} \\ s'_{2} \\ s'_{3} \end{pmatrix} = \begin{pmatrix} 02 & 03 & 01 & 01 \\ 01 & 02 & 03 & 01 \\ 01 & 01 & 02 & 03 \\ 03 & 01 & 01 & 02 \end{pmatrix} \begin{pmatrix} s_{0} \\ s_{1} \\ s_{2} \\ s_{3} \end{pmatrix}$$

- **AddRoundKey**: XOR the state with the round key
  - The only step that uses the key

- **Decryption**: run the **inverse** of each step in reverse order
  - Inverse ShiftRows (rotate right), inverse SubBytes (inverse S-box), AddRoundKey, inverse MixColumns (inverse matrix)
  - Round keys are used in reverse order
  - Unlike Feistel, AES needs separate inverse functions for decryption

#### **AES-128 Execution for the Official Test Vector**

Key `000102030405060708090a0b0c0d0e0f`, plaintext `00112233445566778899aabbccddeeff`. Values are from the Python code below.

| Round | State After the Round              |
|-------|------------------------------------|
| 0     | `00102030405060708090a0b0c0d0e0f0` |
| 1     | `89d810e8855ace682d1843d8cb128fe4` |
| 2     | `4915598f55e5d7a0daca94fa1f0a63f7` |
| ...   | ...                                |
| 9     | `bd6e7c3df2b5779e0b61216e8b10b689` |
| 10    | `69c4e0d86a7b0430d8cdb78070b4c55a` |

Final result:
- Ciphertext: `69c4e0d86a7b0430d8cdb78070b4c55a` (matches the AES standard)
- Avalanche: flipping 1 plaintext bit changes **62 of 128** ciphertext bits

#### **AES vs DES**

| Feature          | DES                        | AES                              |
|------------------|----------------------------|----------------------------------|
| Year             | 1977                       | 2001                             |
| Structure        | Feistel                    | Substitution-permutation network |
| Block size       | 64 bits                    | 128 bits                         |
| Key size         | 56 bits                    | 128, 192 or 256 bits             |
| Rounds           | 16                         | 10, 12 or 14                     |
| Each round changes | Half the block           | The whole block                  |
| Design           | S-box criteria kept secret at first | Fully public, chosen in an open competition |
| Status           | Broken by brute force      | Secure, used everywhere          |



In [2]:
# AES-128 from scratch in Python, to show how each step works.
# For learning only. Use a tested library (see the next cell) for real data.


# ---------- GF(2^8) arithmetic ----------
def gmul(a, b):
    """Multiply two bytes in GF(2^8), reducing by m(x) = x^8 + x^4 + x^3 + x + 1 (0x11B)."""
    result = 0
    while b:
        if b & 1:
            result ^= a          # addition in GF(2^8) is XOR
        a <<= 1                  # multiply a by x
        if a & 0x100:
            a ^= 0x11B           # reduce mod m(x) if degree reaches 8
        b >>= 1
    return result


def ginv(a):
    """Multiplicative inverse in GF(2^8): a^254 = a^-1, since a^255 = 1 for a != 0. Maps 0 to 0."""
    result = 1
    for _ in range(254):
        result = gmul(result, a)
    return result if a else 0


# ---------- S-box: inverse in GF(2^8), then a fixed affine transformation ----------
def build_sbox():
    sbox = []
    for a in range(256):
        b = ginv(a)
        s = b
        for shift in range(1, 5):
            s ^= ((b << shift) | (b >> (8 - shift))) & 0xFF   # rotate left and XOR
        sbox.append(s ^ 0x63)
    return sbox


SBOX = build_sbox()
INV_SBOX = [SBOX.index(i) for i in range(256)]


# ---------- the four round operations (state = 4x4 list, state[row][col]) ----------
def sub_bytes(state, box=SBOX):
    return [[box[b] for b in row] for row in state]


def shift_rows(state, direction=1):
    """Row r is rotated left by r positions (right when decrypting)."""
    return [row[direction * r:] + row[:direction * r] for r, row in enumerate(state)]


def mix_columns(state, matrix=((2, 3, 1, 1), (1, 2, 3, 1), (1, 1, 2, 3), (3, 1, 1, 2))):
    """Multiply each column by a fixed matrix in GF(2^8)."""
    out = [[0] * 4 for _ in range(4)]
    for c in range(4):
        col = [state[r][c] for r in range(4)]
        for r in range(4):
            out[r][c] = gmul(matrix[r][0], col[0]) ^ gmul(matrix[r][1], col[1]) \
                      ^ gmul(matrix[r][2], col[2]) ^ gmul(matrix[r][3], col[3])
    return out


INV_MIX = ((14, 11, 13, 9), (9, 14, 11, 13), (13, 9, 14, 11), (11, 13, 9, 14))


def add_round_key(state, round_key):
    return [[state[r][c] ^ round_key[r][c] for c in range(4)] for r in range(4)]


# ---------- key expansion: 16-byte key -> 44 words -> 11 round keys ----------
RCON = [0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80, 0x1B, 0x36]


def expand_key(key):
    w = [list(key[4 * i:4 * i + 4]) for i in range(4)]            # w[0..3] = the key itself
    for i in range(4, 44):
        temp = w[i - 1][:]
        if i % 4 == 0:
            temp = temp[1:] + temp[:1]                             # RotWord
            temp = [SBOX[b] for b in temp]                         # SubWord
            temp[0] ^= RCON[i // 4 - 1]                            # add round constant
        w.append([a ^ b for a, b in zip(w[i - 4], temp)])
    # group 4 words into a 4x4 round key (each word is one column)
    return [[[w[4 * k + c][r] for c in range(4)] for r in range(4)] for k in range(11)]


# ---------- helpers ----------
def to_state(block):
    """16 bytes fill the 4x4 state column by column."""
    return [[block[r + 4 * c] for c in range(4)] for r in range(4)]


def from_state(state):
    return bytes(state[r][c] for c in range(4) for r in range(4))


# ---------- AES-128 ----------
def encrypt_block(block, key, trace=False):
    keys = expand_key(key)
    state = add_round_key(to_state(block), keys[0])                # initial transformation
    if trace:
        print(f"  Round  0: {from_state(state).hex()}")
    for rnd in range(1, 11):
        state = sub_bytes(state)
        state = shift_rows(state)
        if rnd != 10:                                              # last round skips MixColumns
            state = mix_columns(state)
        state = add_round_key(state, keys[rnd])
        if trace:
            print(f"  Round {rnd:2}: {from_state(state).hex()}")
    return from_state(state)


def decrypt_block(block, key):
    keys = expand_key(key)
    state = add_round_key(to_state(block), keys[10])
    for rnd in range(9, -1, -1):
        state = shift_rows(state, -1)                              # inverse shift rows
        state = sub_bytes(state, INV_SBOX)                         # inverse sub bytes
        state = add_round_key(state, keys[rnd])
        if rnd != 0:
            state = mix_columns(state, INV_MIX)                    # inverse mix columns
    return from_state(state)


# Driver code: official test vector from the AES standard (FIPS-197)
key = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
plaintext = bytes.fromhex("00112233445566778899aabbccddeeff")

print("GF(2^8): {57} x {83} =", hex(gmul(0x57, 0x83)))
print("S-box[0x53] =", hex(SBOX[0x53]))
print("\nEncryption:")
ciphertext = encrypt_block(plaintext, key, trace=True)
print("Cipher text:", ciphertext.hex())
print("Expected:    69c4e0d86a7b0430d8cdb78070b4c55a")
print("Decrypted:  ", decrypt_block(ciphertext, key).hex())

flipped = bytes.fromhex("00112233445566778899aabbccddeefe")  # last bit flipped
c2 = encrypt_block(flipped, key)
diff = bin(int.from_bytes(ciphertext, "big") ^ int.from_bytes(c2, "big")).count("1")
print(f"\nAvalanche: 1 plaintext bit changed -> {diff} of 128 ciphertext bits changed")

GF(2^8): {57} x {83} = 0xc1
S-box[0x53] = 0xed

Encryption:
  Round  0: 00102030405060708090a0b0c0d0e0f0
  Round  1: 89d810e8855ace682d1843d8cb128fe4
  Round  2: 4915598f55e5d7a0daca94fa1f0a63f7
  Round  3: fa636a2825b339c940668a3157244d17
  Round  4: 247240236966b3fa6ed2753288425b6c
  Round  5: c81677bc9b7ac93b25027992b0261996
  Round  6: c62fe109f75eedc3cc79395d84f9cf5d
  Round  7: d1876c0f79c4300ab45594add66ff41f
  Round  8: fde3bad205e5d0d73547964ef1fe37f1
  Round  9: bd6e7c3df2b5779e0b61216e8b10b689
  Round 10: 69c4e0d86a7b0430d8cdb78070b4c55a
Cipher text: 69c4e0d86a7b0430d8cdb78070b4c55a
Expected:    69c4e0d86a7b0430d8cdb78070b4c55a
Decrypted:   00112233445566778899aabbccddeeff

Avalanche: 1 plaintext bit changed -> 62 of 128 ciphertext bits changed


<br>

***